In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [1]:
df_players_raw = spark.table("lh_bronze_game.players_raw")

StatementMeta(, 935784cc-0ce4-4b25-8537-5a0aa79acc72, 3, Finished, Available, Finished, False)

In [2]:

print("Row count:", df_players_raw.count())
print("Columns:", df_players_raw.columns)

df_players_raw.printSchema()
display(df_players_raw.limit(5))


StatementMeta(, 935784cc-0ce4-4b25-8537-5a0aa79acc72, 4, Finished, Available, Finished, False)

Row count: 50000
Columns: ['acquisition_channel', 'age_group', 'campaign_id', 'country', 'creative_experiment_group', 'creative_id', 'device_type', 'experiment_group', 'first_session_timestamp', 'install_date', 'platform', 'player_id']
root
 |-- acquisition_channel: string (nullable = true)
 |-- age_group: string (nullable = true)
 |-- campaign_id: string (nullable = true)
 |-- country: string (nullable = true)
 |-- creative_experiment_group: string (nullable = true)
 |-- creative_id: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- experiment_group: string (nullable = true)
 |-- first_session_timestamp: string (nullable = true)
 |-- install_date: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- player_id: string (nullable = true)



SynapseWidget(Synapse.DataFrame, 7e1be397-d717-4290-bb72-e93a116ef62b)

In [4]:
from pyspark.sql import functions as F
print("total rows:",df_players_raw.count())
print("unique player_id", df_players_raw.select("player_id").distinct().count())

StatementMeta(, 935784cc-0ce4-4b25-8537-5a0aa79acc72, 6, Finished, Available, Finished, False)

total rows: 50000
unique player_id 50000


In [5]:
df_players_raw.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_players_raw.columns
]).show(truncate=False)

StatementMeta(, 935784cc-0ce4-4b25-8537-5a0aa79acc72, 7, Finished, Available, Finished, False)

+-------------------+---------+-----------+-------+-------------------------+-----------+-----------+----------------+-----------------------+------------+--------+---------+
|acquisition_channel|age_group|campaign_id|country|creative_experiment_group|creative_id|device_type|experiment_group|first_session_timestamp|install_date|platform|player_id|
+-------------------+---------+-----------+-------+-------------------------+-----------+-----------+----------------+-----------------------+------------+--------+---------+
|37                 |111      |17541      |0      |0                        |17500      |0          |0               |0                      |0           |0       |0        |
+-------------------+---------+-----------+-------+-------------------------+-----------+-----------+----------------+-----------------------+------------+--------+---------+



In [6]:
df_players_raw.groupBy("acquisition_channel").count().show()

StatementMeta(, 935784cc-0ce4-4b25-8537-5a0aa79acc72, 8, Finished, Available, Finished, False)

+-------------------+-----+
|acquisition_channel|count|
+-------------------+-----+
|            Organic|17500|
|               NULL|   37|
|          Unity Ads| 9306|
|         TikTok Ads| 8691|
|           Meta Ads| 5920|
|         Google Ads| 8546|
+-------------------+-----+



In [7]:
df_players_raw.filter(
    (F.col("acquisition_channel") != "Organic") &
    (
        F.col("campaign_id").isNull() |
        F.col("creative_id").isNull()
    )
).count()

StatementMeta(, 935784cc-0ce4-4b25-8537-5a0aa79acc72, 9, Finished, Available, Finished, False)

41

In [8]:
df_players_clean = (
    df_players_raw.withColumn(
        "campaign_id",
        F.when(
            (F.col("acquisition_channel") != "Organic") &
            F.col("campaign_id").isNull(),
            F.lit("Unknown")
        ).otherwise(F.col("campaign_id"))
    )
    .withColumn(
        "creative_id",
        F.when(
            (F.col("acquisition_channel") != "Organic") &
            F.col("creative_id").isNull(),
            F.lit("Unknown")
        ).otherwise(F.col("creative_id"))
    )
)

StatementMeta(, 935784cc-0ce4-4b25-8537-5a0aa79acc72, 10, Finished, Available, Finished, False)

In [9]:
df_players_clean.filter(
    (F.col("acquisition_channel") != "Organic") &
    (
        F.col("campaign_id").isNull() |
        F.col("creative_id").isNull()
    )
).count()

StatementMeta(, 935784cc-0ce4-4b25-8537-5a0aa79acc72, 11, Finished, Available, Finished, False)

0

In [10]:
total_count = df_players_raw.count()

unique_players = (
    df_players_raw
    .select("player_id")
    .distinct()
    .count()
)

duplicate_count = total_count - unique_players

print("Toplam satır:", total_count)
print("Unique player:", unique_players)
print("Duplicate:", duplicate_count)

StatementMeta(, 935784cc-0ce4-4b25-8537-5a0aa79acc72, 12, Finished, Available, Finished, False)

Toplam satır: 50000
Unique player: 50000
Duplicate: 0


In [11]:
df_players_clean = (
    df_players_clean
    .withColumn(
        "age_group",
        F.when(
            F.col("age_group").isNull(),
            F.lit("Unknown")
        ).otherwise(F.col("age_group"))
    )
    .withColumn(
        "acquisition_channel",
        F.when(
            F.col("acquisition_channel").isNull(),
            F.lit("Unknown")
        ).otherwise(F.col("acquisition_channel"))
    )
)

StatementMeta(, 935784cc-0ce4-4b25-8537-5a0aa79acc72, 13, Finished, Available, Finished, False)

In [12]:
df_players_clean.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in [
        "age_group",
        "acquisition_channel",
        "campaign_id",
        "creative_id"
    ]
]).show()

StatementMeta(, 935784cc-0ce4-4b25-8537-5a0aa79acc72, 14, Finished, Available, Finished, False)

+---------+-------------------+-----------+-----------+
|age_group|acquisition_channel|campaign_id|creative_id|
+---------+-------------------+-----------+-----------+
|        0|                  0|      17500|      17500|
+---------+-------------------+-----------+-----------+



In [13]:
df_players_clean.groupBy("platform").count().show()

df_players_clean.groupBy("country").count().show()

df_players_clean.groupBy("experiment_group").count().show()

df_players_clean.groupBy("creative_experiment_group").count().show()

df_players_clean.groupBy("age_group").count().show()

StatementMeta(, 935784cc-0ce4-4b25-8537-5a0aa79acc72, 15, Finished, Available, Finished, False)

+--------+-----+
|platform|count|
+--------+-----+
|     iOS|21096|
| Android|28904|
+--------+-----+

+--------------+-----+
|       country|count|
+--------------+-----+
|        Turkey|13931|
|       Germany| 7024|
|        France| 3467|
| United States| 2087|
|   South Korea| 1652|
|        Brazil|14046|
|         Japan| 5087|
|United Kingdom| 2706|
+--------------+-----+

+----------------+-----+
|experiment_group|count|
+----------------+-----+
|         Control|25082|
|       Treatment|24918|
+----------------+-----+

+-------------------------+-----+
|creative_experiment_group|count|
+-------------------------+-----+
|               Creative A| 4830|
|           Not Applicable|40340|
|               Creative B| 4830|
+-------------------------+-----+

+---------+-----+
|age_group|count|
+---------+-----+
|    45-54| 6932|
|    13-17| 3013|
|  Unknown|  111|
|    35-44|10886|
|    25-34|14115|
|    18-24| 9851|
|      55+| 5092|
+---------+-----+



In [14]:
df_date_check = (
    df_players_clean
    .withColumn(
        "install_date_parsed",
        F.to_date("install_date", "yyyy-MM-dd")
    )
    .withColumn(
        "first_session_timestamp_parsed",
        F.to_timestamp("first_session_timestamp")
    )
)

df_date_check.select(
    F.count(
        F.when(F.col("install_date_parsed").isNull(), 1)
    ).alias("invalid_install_date"),
    
    F.count(
        F.when(F.col("first_session_timestamp_parsed").isNull(), 1)
    ).alias("invalid_first_session_timestamp")
).show()

StatementMeta(, 935784cc-0ce4-4b25-8537-5a0aa79acc72, 16, Finished, Available, Finished, False)

+--------------------+-------------------------------+
|invalid_install_date|invalid_first_session_timestamp|
+--------------------+-------------------------------+
|                   0|                              0|
+--------------------+-------------------------------+



In [15]:
df_players_clean = (
    df_players_clean
    .withColumn(
        "install_date",
        F.to_date("install_date", "yyyy-MM-dd")
    )
    .withColumn(
        "first_session_timestamp",
        F.to_timestamp("first_session_timestamp")
    )
)

StatementMeta(, 935784cc-0ce4-4b25-8537-5a0aa79acc72, 17, Finished, Available, Finished, False)

In [16]:
df_players_clean.printSchema()

StatementMeta(, 935784cc-0ce4-4b25-8537-5a0aa79acc72, 18, Finished, Available, Finished, False)

root
 |-- acquisition_channel: string (nullable = true)
 |-- age_group: string (nullable = true)
 |-- campaign_id: string (nullable = true)
 |-- country: string (nullable = true)
 |-- creative_experiment_group: string (nullable = true)
 |-- creative_id: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- experiment_group: string (nullable = true)
 |-- first_session_timestamp: timestamp (nullable = true)
 |-- install_date: date (nullable = true)
 |-- platform: string (nullable = true)
 |-- player_id: string (nullable = true)



In [17]:
df_players_clean.filter(
    F.to_date("first_session_timestamp") < F.col("install_date")
).count()

StatementMeta(, 935784cc-0ce4-4b25-8537-5a0aa79acc72, 19, Finished, Available, Finished, False)

0

In [ ]:
df_players_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("players_clean")

In [18]:
df_players_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("lh_silver_game.players_clean")

StatementMeta(, 935784cc-0ce4-4b25-8537-5a0aa79acc72, 20, Finished, Available, Finished, False)

In [19]:
print("Row count:", df_players_clean.count())
print("Unique player_id:", df_players_clean.select("player_id").distinct().count())

df_players_clean.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in [
        "player_id",
        "age_group",
        "acquisition_channel",
        "install_date",
        "first_session_timestamp"
    ]
]).show()

df_players_clean.printSchema()

StatementMeta(, 935784cc-0ce4-4b25-8537-5a0aa79acc72, 21, Finished, Available, Finished, False)

Row count: 50000
Unique player_id: 50000
+---------+---------+-------------------+------------+-----------------------+
|player_id|age_group|acquisition_channel|install_date|first_session_timestamp|
+---------+---------+-------------------+------------+-----------------------+
|        0|        0|                  0|           0|                      0|
+---------+---------+-------------------+------------+-----------------------+

root
 |-- acquisition_channel: string (nullable = true)
 |-- age_group: string (nullable = true)
 |-- campaign_id: string (nullable = true)
 |-- country: string (nullable = true)
 |-- creative_experiment_group: string (nullable = true)
 |-- creative_id: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- experiment_group: string (nullable = true)
 |-- first_session_timestamp: timestamp (nullable = true)
 |-- install_date: date (nullable = true)
 |-- platform: string (nullable = true)
 |-- player_id: string (nullable = true)



In [20]:
df_check = spark.table("lh_silver_game.players_clean")

print("Row count:", df_check.count())
display(df_check.limit(5))

StatementMeta(, 935784cc-0ce4-4b25-8537-5a0aa79acc72, 22, Finished, Available, Finished, False)

Row count: 50000


SynapseWidget(Synapse.DataFrame, f5060d5f-cb44-416a-a370-288ce5d97941)